In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv('heart.csv')
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [ ]:
df.shape

# 918 rows x 12 columns 

(918, 12)

## Remove outliers using z-score 

In [ ]:
from scipy import stats
import numpy as np

# select only numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns

# calculate Z-scores
z = np.abs(stats.zscore(df[numeric_cols]))

# filter out rows where any z-score > 3
df = df[(z < 3).all(axis=1)]
print("After removing outliers:", df.shape)

# reduces rows from 918 -> 890 using z-score


After removing outliers: (890, 12)


## Convert Text Columns to Numbers using Label Encoder and One Hot Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# LabelEncoder to all categorical columns automatically
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

df.head()

# Sex: M=1 F=0
# ChestPainType: ATA=1 NAP=2 ASY=0 
# RestingECG: Normal=1 ST=2 LVH=0
# ExerciseAngina: N=0 Y=1
# ST_Slope: Up=2 Flat=1 Down=0

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,1,1,140,289,0,1,172,0,0.0,2,0
1,49,0,2,160,180,0,1,156,0,1.0,1,1
2,37,1,1,130,283,0,2,98,0,0.0,2,0
3,48,0,0,138,214,0,1,108,1,1.5,1,1
4,54,1,2,150,195,0,1,122,0,0.0,2,0


## Apply Scaling

In [ ]:
# define X and y 
X = df.drop(['HeartDisease'], axis=1)   # axis 1= column, 0=row
y = df['HeartDisease']

X.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,1,1,140,289,0,1,172,0,0.0,2
1,49,0,2,160,180,0,1,156,0,1.0,1
2,37,1,1,130,283,0,2,98,0,0.0,2
3,48,0,0,138,214,0,1,108,1,1.5,1
4,54,1,2,150,195,0,1,122,0,0.0,2


In [ ]:
# scale data 
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# return to dataframe 
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()

# as we can see, the data was scaled

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,-1.420804,0.517500,0.215918,0.474433,0.852559,-0.552255,0.016105,1.380094,-0.814587,-0.860949,1.042505
1,-0.467332,-1.932367,1.260303,1.653561,-0.167203,-0.552255,0.016105,0.748525,-0.814587,0.168258,-0.641396
2,-1.738628,0.517500,0.215918,-0.115131,0.796425,-0.552255,1.608672,-1.540914,-0.814587,-0.860949,1.042505
3,-0.573273,-1.932367,-0.828467,0.356521,0.150888,-0.552255,0.016105,-1.146183,1.227616,0.682862,-0.641396
4,0.062374,0.517500,1.260303,1.063997,-0.026868,-0.552255,0.016105,-0.593560,-0.814587,-0.860949,1.042505


## Build Classification Model 

In [14]:
# split the data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

In [15]:
# for loop to test different models and their accuracy 
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier()
}

results = {}        # store resutls to print 


for name, model in models.items():
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    results[name] = score

# Display results nicely
for name, score in results.items():
    print(f"{name}: {score:.4f}")

SVM: 0.8652
Logistic Regression: 0.8596
Random Forest: 0.8764


All have similar accuracy but Random Forest is the highest 

## PCA to reduce dimensions

In [22]:
# import PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)  # try reducing to 5 components
X_pca = pca.fit_transform(X_scaled)

In [23]:
# train data 
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y, test_size=0.2)

In [24]:
# view PCA model accuracies
pca_results = {}        # store resutls to print 


for name, model in models.items():
    model.fit(X_train_pca, y_train_pca)
    score = model.score(X_test_pca, y_test_pca)
    pca_results[name] = score

print('Model Scores after PCA:')

# Display results nicely
for name, score in pca_results.items():
    print(f"{name}: {score:.4f}")

Model Scores after PCA:
SVM: 0.7809
Logistic Regression: 0.7865
Random Forest: 0.7697


## As we can see, we reduced our model to 2 dimensions and we got ~0.1 difference in model accuracy. In this example, PCA was reduced significantly so we see a noticeable change in the score.